# 🔗 Chain Snatching Detector v2 — Sentinel AI
**Fixes applied:** Temporal grab confirmation · Scale-aware velocity · False positive mitigation · Camera calibration

## Cell 1 — Imports

In [ ]:
import cv2
import numpy as np
import time
import matplotlib.pyplot as plt
from ultralytics import YOLO
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple
from collections import deque
from IPython.display import clear_output

print('✅ Imports OK')

## Cell 2 — Constants & Keypoint Map

In [ ]:
# COCO-17 keypoint indices (YOLOv8-pose output order)
KP = {
    'nose': 0,
    'left_eye': 1,  'right_eye': 2,
    'left_ear': 3,  'right_ear': 4,
    'left_shoulder': 5,   'right_shoulder': 6,
    'left_elbow': 7,      'right_elbow': 8,
    'left_wrist': 9,      'right_wrist': 10,
    'left_hip': 11,       'right_hip': 12,
    'left_knee': 13,      'right_knee': 14,
    'left_ankle': 15,     'right_ankle': 16,
}

# ── Tunable thresholds ────────────────────────────────────────────────
CFG = {
    # Grab detection
    'GRAB_DIST_PX'         : 80,    # Raw pixel grab radius (pre-scale)
    'GRAB_CONFIRM_FRAMES'  : 3,     # Min frames grab must persist to confirm
    'GRAB_WINDOW'          : 5,     # Rolling window size for grab history

    # Fall detection
    'FALL_ASPECT_RATIO'    : 1.6,   # bbox w/h > this → person horizontal
    'FALL_CONFIRM_FRAMES'  : 3,     # Consecutive fallen frames needed

    # Movement
    'ESCAPE_SPEED_MPS'     : 2.5,   # Metres/second to qualify as fleeing
    'VICTIM_STATIC_MPS'    : 0.4,   # Victim speed below this → likely restrained

    # Scene
    'PROXIMITY_PX'         : 200,   # Max person-centre distance for interaction
    'KP_CONF_THRESH'       : 0.30,  # Ignore keypoints below this confidence

    # Scoring
    'FIRE_THRESHOLD'       : 0.45,  # Minimum score to raise alert
    'ALERT_COOLDOWN_S'     : 3.0,   # Seconds between successive alerts
    'TRACK_TIMEOUT_S'      : 2.0,   # Prune tracks idle this long

    # Physics
    'AVG_PERSON_HEIGHT_M'  : 1.70,  # Used for px-per-metre estimation
    'AVG_SHOULDER_WIDTH_M' : 0.45,  # Used for scale normalisation
    'VIDEO_FPS'            : 25,    # Override if known; auto-detected from cap
}

print('✅ Config ready')

## Cell 3 — GrabState (Temporal Confirmation)

In [ ]:
class GrabState:
    """
    Tracks whether a grab is happening across a rolling window of frames.
    A grab is only *confirmed* when it appears in >= GRAB_CONFIRM_FRAMES
    out of the last GRAB_WINDOW frames — eliminates single-frame handshakes.
    """
    def __init__(self):
        self.history: deque = deque(maxlen=CFG['GRAB_WINDOW'])
        self.min_dist_history: deque = deque(maxlen=CFG['GRAB_WINDOW'])
        self.confirmed: bool = False
        self.raw_score: float = 0.0

    def update(self, min_wrist_to_neck_px: float, grab_radius_px: float):
        is_grab = min_wrist_to_neck_px < grab_radius_px
        self.history.append(is_grab)
        self.min_dist_history.append(min_wrist_to_neck_px)

        grab_count     = sum(self.history)
        self.confirmed = grab_count >= CFG['GRAB_CONFIRM_FRAMES']

        # Smooth score: fraction of recent frames that showed grab
        if self.history:
            frac = grab_count / len(self.history)
            # Also weight by how close the wrist was
            if self.min_dist_history and grab_radius_px > 0:
                avg_dist = float(np.mean(list(self.min_dist_history)))
                closeness = max(0.0, 1.0 - avg_dist / grab_radius_px)
                self.raw_score = frac * closeness
            else:
                self.raw_score = frac
        else:
            self.raw_score = 0.0

    def reset(self):
        self.history.clear()
        self.min_dist_history.clear()
        self.confirmed = False
        self.raw_score = 0.0


print('✅ GrabState defined')

## Cell 4 — PersonTrack (Scale-Aware)

In [ ]:
class PersonTrack:
    """
    Rolling history for one tracked person.
    Velocity is converted to metres/second using estimated pixel scale.
    """

    def __init__(self, track_id: int, fps: float = 25.0):
        self.track_id        = track_id
        self.fps             = fps
        self.bbox_history    : deque = deque(maxlen=15)
        self.keypoint_history: deque = deque(maxlen=15)
        self.last_seen       : float = time.time()
        self._fall_count     : int   = 0
        # Per-pair grab states keyed by victim track_id
        self.grab_states     : Dict[int, GrabState] = {}

    # ── Update ────────────────────────────────────────────────────────

    def update(self, bbox: np.ndarray, keypoints: np.ndarray):
        self.bbox_history.append(bbox.copy())
        self.keypoint_history.append(keypoints.copy())
        self.last_seen = time.time()

        # Fall counter update
        if self._current_aspect_ratio > CFG['FALL_ASPECT_RATIO']:
            self._fall_count = min(self._fall_count + 1, 10)
        else:
            self._fall_count = max(self._fall_count - 1, 0)

    # ── Geometry helpers ──────────────────────────────────────────────

    @property
    def center(self) -> Optional[np.ndarray]:
        if not self.bbox_history:
            return None
        b = self.bbox_history[-1]
        return np.array([(b[0] + b[2]) / 2, (b[1] + b[3]) / 2])

    @property
    def _current_aspect_ratio(self) -> float:
        if not self.bbox_history:
            return 0.0
        b = self.bbox_history[-1]
        h = b[3] - b[1]
        w = b[2] - b[0]
        return (w / h) if h > 0 else 0.0

    @property
    def is_fallen(self) -> bool:
        return self._fall_count >= CFG['FALL_CONFIRM_FRAMES']

    # ── Scale estimation ──────────────────────────────────────────────

    @property
    def px_per_metre(self) -> float:
        """
        Estimate pixels-per-metre from bounding box height.
        Assumes average person = 1.70 m tall.
        Falls back to shoulder-width method if keypoints available.
        """
        # Method 1: Shoulder width (more stable for partial bodies)
        if self.keypoint_history:
            kps = self.keypoint_history[-1]
            ls  = kps[KP['left_shoulder']]
            rs  = kps[KP['right_shoulder']]
            if ls[2] > CFG['KP_CONF_THRESH'] and rs[2] > CFG['KP_CONF_THRESH']:
                shoulder_px = float(np.linalg.norm(
                    np.array([ls[0], ls[1]]) - np.array([rs[0], rs[1]])
                ))
                if shoulder_px > 5:
                    return shoulder_px / CFG['AVG_SHOULDER_WIDTH_M']

        # Method 2: Bounding box height
        if self.bbox_history:
            b  = self.bbox_history[-1]
            h  = b[3] - b[1]
            if h > 5:
                return h / CFG['AVG_PERSON_HEIGHT_M']

        return 100.0  # Fallback: assume 100 px/m

    @property
    def grab_radius_px(self) -> float:
        """Scale grab threshold to person's apparent size."""
        # The raw threshold assumes ~100 px/m reference distance
        return CFG['GRAB_DIST_PX'] * (self.px_per_metre / 100.0)

    # ── Velocity ──────────────────────────────────────────────────────

    @property
    def velocity_mps(self) -> float:
        """
        Speed in metres per second, accounting for camera perspective.
        velocity_mps = (px_displacement_per_frame * fps) / px_per_metre
        """
        if len(self.bbox_history) < 2:
            return 0.0
        recent = list(self.bbox_history)[-5:]
        centers = [np.array([(b[0]+b[2])/2, (b[1]+b[3])/2]) for b in recent]
        px_per_frame = float(np.mean([
            np.linalg.norm(centers[i+1] - centers[i])
            for i in range(len(centers)-1)
        ]))
        ppm = max(self.px_per_metre, 1.0)
        return (px_per_frame * self.fps) / ppm

    # ── Keypoint accessors ────────────────────────────────────────────

    @property
    def neck_position(self) -> Optional[np.ndarray]:
        """Midpoint of shoulders — where a chain would sit."""
        if not self.keypoint_history:
            return None
        kps = self.keypoint_history[-1]
        ls  = kps[KP['left_shoulder']]
        rs  = kps[KP['right_shoulder']]
        if ls[2] < CFG['KP_CONF_THRESH'] or rs[2] < CFG['KP_CONF_THRESH']:
            return None
        return np.array([(ls[0]+rs[0])/2, (ls[1]+rs[1])/2])

    def wrist_positions(self) -> List[np.ndarray]:
        if not self.keypoint_history:
            return []
        kps = self.keypoint_history[-1]
        out = []
        for side in ['left_wrist', 'right_wrist']:
            w = kps[KP[side]]
            if w[2] > CFG['KP_CONF_THRESH']:
                out.append(np.array([w[0], w[1]]))
        return out

    def get_grab_state(self, victim_id: int) -> GrabState:
        if victim_id not in self.grab_states:
            self.grab_states[victim_id] = GrabState()
        return self.grab_states[victim_id]


print('✅ PersonTrack defined')

## Cell 5 — ThreatEvent Dataclass

In [ ]:
@dataclass
class ThreatEvent:
    label        : str
    confidence   : float          # 0.0 – 1.0
    severity     : str            # LOW / MEDIUM / HIGH / CRITICAL
    frame_number : int
    attacker_id  : Optional[int]
    victim_id    : Optional[int]
    details      : Dict = field(default_factory=dict)

    def to_sentinel_payload(self) -> dict:
        """Converts to Sentinel AI fusion-layer compatible dict."""
        return {
            'threat_type'  : self.label,
            'confidence'   : round(self.confidence, 3),
            'severity'     : self.severity,
            'frame_number' : self.frame_number,
            'attacker_id'  : self.attacker_id,
            'victim_id'    : self.victim_id,
            'details'      : self.details,
        }

print('✅ ThreatEvent defined')

## Cell 6 — Main Detector Class

In [ ]:
class ChainSnatchingDetector:
    """
    Detects chain-snatching from video frames using YOLOv8-pose.

    Four signals (all required to confirm):
      1. Temporal grab   — wrist near victim neck for 3+ consecutive frames
      2. Victim fall     — bounding box goes horizontal (confirmed over 3 frames)
      3. Context check   — victim is slow (restrained), attacker is fast (fleeing)
      4. Proximity guard — pair must be within interaction distance

    Usage
    -----
    detector = ChainSnatchingDetector(fps=25)
    event = detector.process_frame(bgr_frame, frame_number=42)
    annotated = detector.annotate_frame(bgr_frame, event)
    """

    def __init__(self, model_path: str = 'yolov8x-pose.pt',
                 device: str = 'cpu', fps: float = 25.0):
        print('⏳ Loading YOLOv8-pose model…')
        self.model           = YOLO(model_path)
        self.device          = device
        self.fps             = fps
        self.tracks          : Dict[int, PersonTrack] = {}
        self._last_alert_t   : float = 0.0
        self._frame_id       : int   = 0
        print('✅ Model ready.')

    # ─────────────────────────────────────────────────────────────────
    #  Public API
    # ─────────────────────────────────────────────────────────────────

    def process_frame(self,
                      frame: np.ndarray,
                      frame_number: Optional[int] = None
                     ) -> Optional[ThreatEvent]:
        self._frame_id = frame_number if frame_number is not None else self._frame_id + 1

        # ── Inference ──────────────────────────────────────────────
        results = self.model.track(
            frame, persist=True, verbose=False,
            device=self.device, conf=0.35
        )
        if not results or results[0].boxes is None:
            return None
        result    = results[0]
        boxes     = result.boxes
        keypoints = result.keypoints
        if keypoints is None:
            return None

        ids    = (boxes.id.cpu().numpy().astype(int)
                  if boxes.id is not None else np.arange(len(boxes)))
        bboxes = boxes.xyxy.cpu().numpy()
        kps    = keypoints.data.cpu().numpy()   # (N, 17, 3)

        # ── Update tracks ──────────────────────────────────────────
        self._prune_old_tracks()
        for i, tid in enumerate(ids):
            if tid not in self.tracks:
                self.tracks[tid] = PersonTrack(track_id=int(tid), fps=self.fps)
            self.tracks[tid].update(bboxes[i], kps[i])

        # ── Score pairs ────────────────────────────────────────────
        event = self._evaluate_all_pairs()

        if event and (time.time() - self._last_alert_t) >= CFG['ALERT_COOLDOWN_S']:
            self._last_alert_t = time.time()
            return event
        return None

    def annotate_frame(self,
                       frame: np.ndarray,
                       event: Optional[ThreatEvent] = None
                      ) -> np.ndarray:
        out = frame.copy()
        attacker_id = event.attacker_id if event else None
        victim_id   = event.victim_id   if event else None

        COLOURS = {
            'normal'  : (50, 200, 50),
            'attacker': (0, 0, 255),
            'victim'  : (0, 140, 255),
        }

        for tid, track in self.tracks.items():
            if not track.bbox_history:
                continue
            b = track.bbox_history[-1].astype(int)

            if tid == attacker_id:
                col, lbl = COLOURS['attacker'], f'ATTACKER [{tid}]'
            elif tid == victim_id:
                col, lbl = COLOURS['victim'], f'VICTIM [{tid}]'
            else:
                col, lbl = COLOURS['normal'], f'ID {tid}'

            speed_txt = f'  {track.velocity_mps:.1f}m/s'
            cv2.rectangle(out, (b[0], b[1]), (b[2], b[3]), col, 2)
            cv2.putText(out, lbl + speed_txt,
                        (b[0], b[1] - 8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, col, 2)

        if event:
            self._draw_alert_banner(out, event)
        return out

    # ─────────────────────────────────────────────────────────────────
    #  Scoring
    # ─────────────────────────────────────────────────────────────────

    def _evaluate_all_pairs(self) -> Optional[ThreatEvent]:
        ids = list(self.tracks.keys())
        if len(ids) < 2:
            return None

        best_score, best_event = 0.0, None

        for i in range(len(ids)):
            for j in range(len(ids)):
                if i == j:
                    continue
                att = self.tracks[ids[i]]
                vic = self.tracks[ids[j]]
                score, details = self._score_pair(att, vic)

                if score > best_score:
                    best_score = score
                    best_event = ThreatEvent(
                        label        = 'CHAIN_SNATCHING',
                        confidence   = min(score, 1.0),
                        severity     = self._to_severity(score),
                        frame_number = self._frame_id,
                        attacker_id  = att.track_id,
                        victim_id    = vic.track_id,
                        details      = details,
                    )

        return best_event if best_score >= CFG['FIRE_THRESHOLD'] else None

    def _score_pair(self,
                    att: PersonTrack,
                    vic: PersonTrack
                   ) -> Tuple[float, Dict]:
        """
        Score one attacker–victim pair.

        Weights
        -------
        Temporal grab (confirmed)  → 0.45
        Victim fall (confirmed)    → 0.25
        Context: victim static     → 0.15
        Context: attacker fleeing  → 0.10
        Proximity                  → 0.05
        """
        score   = 0.0
        details = {}

        ac, vc = att.center, vic.center
        if ac is None or vc is None:
            return 0.0, details

        # ── Guard: proximity ──────────────────────────────────────
        person_dist = float(np.linalg.norm(ac - vc))
        details['person_dist_px'] = round(person_dist, 1)
        if person_dist > CFG['PROXIMITY_PX']:
            return 0.0, details
        prox_score = max(0.0, 1.0 - person_dist / CFG['PROXIMITY_PX'])
        score += prox_score * 0.05
        details['proximity_score'] = round(prox_score, 2)

        # ── Signal 1: Temporal grab ───────────────────────────────
        victim_neck = vic.neck_position
        grab_state  = att.get_grab_state(vic.track_id)

        if victim_neck is not None and att.wrist_positions():
            min_dist = min(
                float(np.linalg.norm(w - victim_neck))
                for w in att.wrist_positions()
            )
            grab_state.update(min_dist, att.grab_radius_px)
            details['wrist_neck_dist_px']  = round(min_dist, 1)
            details['grab_radius_px']      = round(att.grab_radius_px, 1)
            details['grab_confirmed']      = grab_state.confirmed
            details['grab_score']          = round(grab_state.raw_score, 2)

            if grab_state.confirmed:
                score += grab_state.raw_score * 0.45
        else:
            grab_state.reset()

        # ── Signal 2: Victim fall ─────────────────────────────────
        details['victim_fallen'] = vic.is_fallen
        if vic.is_fallen:
            score += 0.25

        # ── Signal 3: Victim static (restrained / pulled down) ────
        vic_speed = vic.velocity_mps
        details['victim_speed_mps'] = round(vic_speed, 2)
        victim_static = vic_speed < CFG['VICTIM_STATIC_MPS']
        details['victim_static'] = victim_static

        # Apply context multiplier to grab score (not full score)
        # If victim is moving fast → likely a friendly interaction → reduce
        if grab_state.confirmed:
            if victim_static:
                score += 0.15                    # bonus: victim restrained
            else:
                score = max(0.0, score - 0.15)   # penalty: probably friendly

        # ── Signal 4: Attacker escape velocity ────────────────────
        att_speed = att.velocity_mps
        details['attacker_speed_mps'] = round(att_speed, 2)
        details['escape_threshold_mps'] = CFG['ESCAPE_SPEED_MPS']

        if att_speed >= CFG['ESCAPE_SPEED_MPS']:
            escape_score = min(1.0, (att_speed - CFG['ESCAPE_SPEED_MPS']) / 3.0)
            score += escape_score * 0.10
            details['escape_score'] = round(escape_score, 2)

        return score, details

    # ─────────────────────────────────────────────────────────────────
    #  Helpers
    # ─────────────────────────────────────────────────────────────────

    @staticmethod
    def _to_severity(score: float) -> str:
        if score >= 0.80: return 'CRITICAL'
        if score >= 0.65: return 'HIGH'
        if score >= 0.45: return 'MEDIUM'
        return 'LOW'

    def _draw_alert_banner(self, frame: np.ndarray, event: ThreatEvent):
        h, w  = frame.shape[:2]
        COLS  = {'CRITICAL': (0,0,180), 'HIGH': (0,0,220),
                 'MEDIUM': (0,100,220), 'LOW': (30,150,255)}
        c     = COLS.get(event.severity, (0, 0, 255))
        ov    = frame.copy()
        cv2.rectangle(ov, (0, 0), (w, 65), c, -1)
        cv2.addWeighted(ov, 0.55, frame, 0.45, 0, frame)
        txt = (f'CHAIN SNATCHING  |  {event.severity}  '
               f'|  Conf: {event.confidence:.0%}  '
               f'|  Frame: {event.frame_number}')
        cv2.putText(frame, txt, (12, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.72, (255,255,255), 2)

    def _prune_old_tracks(self):
        now   = time.time()
        stale = [tid for tid, t in self.tracks.items()
                 if now - t.last_seen > CFG['TRACK_TIMEOUT_S']]
        for tid in stale:
            del self.tracks[tid]


print('✅ ChainSnatchingDetector v2 defined')

## Cell 7 — Run on Video File

In [ ]:
# ── Config ───────────────────────────────────────────────────────────
VIDEO_PATH    = 'test_video.mp4'     # ← your video
OUTPUT_PATH   = 'output_v2.mp4'
SAVE_OUTPUT   = True
PREVIEW_EVERY = 15                   # Show frame every N frames in notebook
# ─────────────────────────────────────────────────────────────────────

cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise FileNotFoundError(f'Cannot open: {VIDEO_PATH}')

fps    = cap.get(cv2.CAP_PROP_FPS) or 25.0
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
CFG['VIDEO_FPS'] = fps   # propagate to config

detector   = ChainSnatchingDetector(fps=fps)
events_log = []

writer = None
if SAVE_OUTPUT:
    writer = cv2.VideoWriter(
        OUTPUT_PATH,
        cv2.VideoWriter_fourcc(*'mp4v'),
        fps, (width, height)
    )

frame_num = 0
print(f'📹 {VIDEO_PATH}  |  {total} frames  |  {fps:.1f} fps  |  {width}×{height}')

while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame_num += 1

    event     = detector.process_frame(frame, frame_number=frame_num)
    annotated = detector.annotate_frame(frame, event)

    if event:
        events_log.append(event)
        print(f'  🚨  Frame {frame_num:>5}  {event.severity:<8}  '
              f'conf={event.confidence:.0%}  '
              f'att_speed={event.details.get("attacker_speed_mps","?"):.1f}m/s  '
              f'vic_fallen={event.details.get("victim_fallen", False)}')

    if writer:
        writer.write(annotated)

    if frame_num % PREVIEW_EVERY == 0:
        rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
        clear_output(wait=True)
        plt.figure(figsize=(13, 6))
        plt.imshow(rgb)
        plt.title(f'Frame {frame_num}/{total}  |  Events: {len(events_log)}')
        plt.axis('off')
        plt.tight_layout()
        plt.show()

cap.release()
if writer:
    writer.release()

print(f'\n✅ Done.  Total events: {len(events_log)}')
if SAVE_OUTPUT:
    print(f'📁 Saved: {OUTPUT_PATH}')

## Cell 8 — Events Summary & Analysis

In [ ]:
if not events_log:
    print('No events detected.')
else:
    severities = [e.severity for e in events_log]
    print(f'\n📊 Detection Summary')
    print(f'   Total events : {len(events_log)}')
    for sev in ['CRITICAL', 'HIGH', 'MEDIUM', 'LOW']:
        n = severities.count(sev)
        if n: print(f'   {sev:<10}: {n}')
    print(f'   Avg conf     : {np.mean([e.confidence for e in events_log]):.0%}')

    print('\n   Detailed timeline:')
    for e in events_log:
        d = e.details
        print(f'     Frame {e.frame_number:>5}  {e.severity:<8}  '
              f'conf={e.confidence:.0%}  '
              f'grab={d.get("grab_confirmed",False)}  '
              f'fall={d.get("victim_fallen",False)}  '
              f'att={d.get("attacker_speed_mps",0):.1f}m/s  '
              f'vic={d.get("victim_speed_mps",0):.1f}m/s')

    # Confidence distribution chart
    confs = [e.confidence for e in events_log]
    frames = [e.frame_number for e in events_log]
    plt.figure(figsize=(12, 3))
    plt.bar(frames, confs, color='crimson', width=2)
    plt.axhline(CFG['FIRE_THRESHOLD'], linestyle='--', color='orange', label='Fire threshold')
    plt.xlabel('Frame'); plt.ylabel('Confidence')
    plt.title('Chain Snatching Confidence Over Time')
    plt.legend(); plt.tight_layout(); plt.show()

## Cell 9 — Sentinel AI Integration Snippet
Paste into `detection/video_detector.py`

In [ ]:
# ── detection/video_detector.py (add these lines) ────────────────────
#
# from detection.chain_snatch import ChainSnatchingDetector
#
# _snatch_detector = ChainSnatchingDetector(fps=25)
#
# async def handle_video_frame(frame_b64: str, camera_id: str) -> dict | None:
#     img_bytes = base64.b64decode(frame_b64)
#     arr       = np.frombuffer(img_bytes, np.uint8)
#     frame     = cv2.imdecode(arr, cv2.IMREAD_COLOR)
#
#     event = _snatch_detector.process_frame(frame)
#     if event is None:
#         return None
#
#     payload = event.to_sentinel_payload()
#     payload['camera_id'] = camera_id
#     return payload   # → pass to fusion.py
# ─────────────────────────────────────────────────────────────────────
print('Integration snippet ready — see comments above.')